In [ ]:
from typing import TypedDict
from rich import print
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import InMemorySaver


class OverAllState(TypedDict):
    username: str
    age: int


def node_a(state: OverAllState) -> OverAllState:
    username = interrupt("お名前を入力してください")
    return {
        "username": username
    }


def node_b(state: OverAllState) -> OverAllState:
    age = interrupt("年齢を入力してください")
    return {
        "age": age
    }


builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge(START, "node_b")
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

from IPython.display import display

display(graph)

config = {"configurable": {"thread_id": "123"}}
interrupted_res = graph.invoke({}, config=config)
print('=' * 30, '-> interrupt_res <-', '=' * 30)
print(interrupted_res)

print(list(graph.get_state_history(config=config)))

In [ ]:
# 6. 実行を再開
resume_map = {}
for i in interrupted_res['__interrupt__']:
    user_input = input(f"{i.value}:")
    if "年齢" in i.value:
        resume_map[i.id] = int(user_input)
    else:
        resume_map[i.id] = user_input

resumed_res = graph.invoke(Command(resume=resume_map), config=config)
print(resumed_res)

In [ ]:
print(list(graph.get_state_history(config=config)))